In [23]:
import pandas as pd

file_path = "../data/cleaned/wine_data_cleaned.csv"

df = pd.read_csv(file_path)


print(f"Shape : {df.shape}")
print(f"\nColonnes : {df.columns.tolist()}")
print(f"\nAperçu :")
print(df.head())



Shape : (54170, 12)

Colonnes : ['id', 'country', 'description', 'designation', 'points', 'price', 'province', 'primary_region', 'taster_name', 'title', 'variety', 'winery']

Aperçu :
   id country                                        description  \
0   3      US  Pineapple rind, lemon pith and orange blossom ...   
1   4      US  Much like the regular bottling from 2012, this...   
2   5   Spain  Blackberry and raspberry aromas show a typical...   
3   6   Italy  Here's a bright, informal red that opens with ...   
4   9  France  This has great depth of flavor with its fresh ...   

                          designation  points  price           province  \
0                Reserve Late Harvest      87   13.0           Michigan   
1  Vintner's Reserve Wild Child Block      87   65.0             Oregon   
2                        Ars In Vitro      87   15.0     Northern Spain   
3                             Belsito      87   16.0  Sicily & Sardinia   
4                         Les Na

In [24]:
y = df['points']
X = df.drop(['points', 'id', 'description', 'title'], axis=1)

# Identifier les types de colonnes
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print("\n" + "="*60)
print("ANALYSE DES FEATURES")
print("="*60)
print(f"\nFeatures numériques ({len(numeric_features)}) :")
for feat in numeric_features:
    print(f"   - {feat}")

print(f"\nFeatures catégorielles ({len(categorical_features)}) :")
for feat in categorical_features:
    unique_count = X[feat].nunique()
    print(f"   - {feat}: {unique_count} valeurs uniques")



ANALYSE DES FEATURES

Features numériques (1) :
   - price

Features catégorielles (7) :
   - country: 7 valeurs uniques
   - designation: 23943 valeurs uniques
   - province: 63 valeurs uniques
   - primary_region: 1004 valeurs uniques
   - taster_name: 17 valeurs uniques
   - variety: 434 valeurs uniques
   - winery: 8583 valeurs uniques


In [25]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# Charger les données
file_path = "../data/cleaned/wine_data_cleaned.csv"
df = pd.read_csv(file_path)

# Définir X et y
y = df['points']
X = df.drop(['points', 'id', 'description', 'title'], axis=1, errors='ignore')

print(f"Données initiales : {X.shape}")
print(f"Colonnes : {X.columns.tolist()}")

# # Identifier les colonnes catégorielles
# categorical_features = X.select_dtypes(include=['object']).columns.tolist()
# numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
#
# print(f"\nFeatures numériques ({len(numeric_features)}) : {numeric_features}")
# print(f"Features catégorielles ({len(categorical_features)}) : {categorical_features}")
#
# # Copier X
# X_encoded = X.copy()
#
# # LabelEncoder sur TOUTES les colonnes catégorielles
# print(f"\nLabelEncoder en cours...")
#
# label_encoders = {}
#
# for col in categorical_features:
#     le = LabelEncoder()
#     X_encoded[col] = le.fit_transform(X_encoded[col].astype(str)).astype(float)
#     label_encoders[col] = le
#
#     print(f"   ✓ {col}: {len(le.classes_)} catégories → encodées de 0 à {len(le.classes_)-1}")
#
# print(f"\nEncodage terminé !")
# print(f"Shape : {X_encoded.shape} (inchangée)")
#
# # Vérifier qu'il n'y a plus de colonnes 'object'
# print(f"\nTypes de données finaux :")
# print(X_encoded.dtypes.value_counts())
#
# # Vérifier les NaN
# nan_count = X_encoded.isnull().sum().sum()
# print(f"\nValeurs manquantes : {nan_count}")
#
# # Aperçu
# print(f"\nAperçu des 5 premières lignes :")
# print(X_encoded.head())
#
# print(f"\nPrêt pour le split train/val/test !")
# print(f"   X : {X_encoded.shape}")
# print(f"   y : {y.shape}")

Données initiales : (54170, 8)
Colonnes : ['country', 'designation', 'price', 'province', 'primary_region', 'taster_name', 'variety', 'winery']


In [26]:
# Séparer en deux groupes
low_cardinality = [col for col in categorical_features if X[col].nunique() < 100]
high_cardinality = [col for col in categorical_features if X[col].nunique() >= 100]

print(f"🔹 OneHotEncoder sur ({len(low_cardinality)}) : {low_cardinality}")
print(f"🔹 LabelEncoder sur ({len(high_cardinality)}) : {high_cardinality}")

# OneHot pour low cardinality
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_onehot = encoder.fit_transform(X[low_cardinality])
onehot_names = encoder.get_feature_names_out(low_cardinality)
X_onehot_df = pd.DataFrame(X_onehot, columns=onehot_names, index=X.index)

# Label pour high cardinality
from sklearn.preprocessing import LabelEncoder
X_label = X[high_cardinality].copy()
for col in high_cardinality:
    le = LabelEncoder()
    X_label[col] = le.fit_transform(X_label[col].astype(str))

# Combiner
X_encoded = pd.concat([X[numeric_features], X_label, X_onehot_df], axis=1)

print(f"\n✅ Approche mixte :")
print(f"   Features finales : {X_encoded.shape[1]}")

🔹 OneHotEncoder sur (3) : ['country', 'province', 'taster_name']
🔹 LabelEncoder sur (4) : ['designation', 'primary_region', 'variety', 'winery']

✅ Approche mixte :
   Features finales : 92


### Segmentation des données

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X_encoded,
    y,
    test_size=0.4,      # 40% pour temp (validation + test)
    random_state=42      # Pour la reproductibilité
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,      # 50% de 40% = 20% du total
    random_state=42
)

# Afficher les résultats
print("="*60)
print("SPLIT DES DONNÉES")
print("="*60)

print(f"\nTailles des ensembles :")
print(f"   Train      : {X_train.shape} - {len(X_train)} lignes ({len(X_train)/len(X_encoded)*100:.1f}%)")
print(f"   Validation : {X_val.shape} - {len(X_val)} lignes ({len(X_val)/len(X_encoded)*100:.1f}%)")
print(f"   Test       : {X_test.shape} - {len(X_test)} lignes ({len(X_test)/len(X_encoded)*100:.1f}%)")

print(f"\nTarget (y) :")
print(f"   y_train : {y_train.shape}")
print(f"   y_val   : {y_val.shape}")
print(f"   y_test  : {y_test.shape}")



SPLIT DES DONNÉES

Tailles des ensembles :
   Train      : (32502, 92) - 32502 lignes (60.0%)
   Validation : (10834, 92) - 10834 lignes (20.0%)
   Test       : (10834, 92) - 10834 lignes (20.0%)

Target (y) :
   y_train : (32502,)
   y_val   : (10834,)
   y_test  : (10834,)


### Normalisation des données

In [28]:
from sklearn.preprocessing import StandardScaler
import pandas as pd


# Créer le scaler
scaler = StandardScaler()

scaler.fit(X_train)

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Normalisation effectuée")


X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_val_scaled = pd.DataFrame(
    X_val_scaled,
    columns=X_val.columns,
    index=X_val.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

print("Reconverti en DataFrame")
print(f"\nAperçu de X_train_scaled :")
print(X_train_scaled.head())

Normalisation effectuée
Reconverti en DataFrame

Aperçu de X_train_scaled :
          price  designation  primary_region   variety    winery  \
42155 -0.527445     1.619330        0.686642  1.116981 -1.449587   
12715 -0.527445    -1.072520        0.593031 -0.814757 -1.324586   
48211  0.437380    -1.687231        0.915828  0.405713 -1.497572   
30440 -0.251780     1.088776        0.973932 -1.315877 -1.193133   
43153  0.850876    -0.313644       -0.575497  0.405713  1.211730   

       country_Argentina  country_Australia  country_Canada  country_France  \
42155          -0.231924          -0.166819       -0.056658       -0.522533   
12715          -0.231924          -0.166819       -0.056658       -0.522533   
48211          -0.231924          -0.166819       -0.056658       -0.522533   
30440          -0.231924          -0.166819       -0.056658       -0.522533   
43153          -0.231924          -0.166819       -0.056658       -0.522533   

       country_Italy  ...  taster_name_J

### Entrainement du modèle

In [73]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
import time

print("="*80)
print("TEST DE MULTIPLES CONFIGURATIONS RANDOM FOREST")
print("="*80)

# Définir les différentes configurations à tester
configurations = [
    {
        'name': 'Config 1 - Baseline',
        'n_estimators': 100,
        'max_depth': 20,
        'min_samples_split': 10,
        'min_samples_leaf': 5,
        'max_features': 'sqrt'
    },
    {
        'name': 'Config 2 - Plus d\'arbres',
        'n_estimators': 200,
        'max_depth': 20,
        'min_samples_split': 10,
        'min_samples_leaf': 5,
        'max_features': 'sqrt'
    },
    {
        'name': 'Config 3 - Arbres profonds',
        'n_estimators': 200,
        'max_depth': 30,
        'min_samples_split': 10,
        'min_samples_leaf': 5,
        'max_features': 'sqrt'
    },
    {
        'name': 'Config 4 - Moins restrictif',
        'n_estimators': 200,
        'max_depth': 30,
        'min_samples_split': 5,
        'min_samples_leaf': 2,
        'max_features': 'sqrt'
    },
    {
        'name': 'Config 5 - Plus restrictif',
        'n_estimators': 200,
        'max_depth': 30,
        'min_samples_split': 15,
        'min_samples_leaf': 8,
        'max_features': 'sqrt'
    },
    {
        'name': 'Config 6 - Max features log2',
        'n_estimators': 200,
        'max_depth': 30,
        'min_samples_split': 10,
        'min_samples_leaf': 5,
        'max_features': 'log2'
    },
    {
        'name': 'Config 7 - Votre config',
        'n_estimators': 200,
        'max_depth': 30,
        'min_samples_split': 13,
        'min_samples_leaf': 4,
        'max_features': 'sqrt'
    },
    {
        'name': 'Config 8 - Beaucoup d\'arbres',
        'n_estimators': 300,
        'max_depth': 25,
        'min_samples_split': 10,
        'min_samples_leaf': 5,
        'max_features': 'sqrt'
    },
    {
        'name': 'Config 9 - Très profond',
        'n_estimators': 150,
        'max_depth': 40,
        'min_samples_split': 10,
        'min_samples_leaf': 5,
        'max_features': 'sqrt'
    },
    {
        'name': 'Config 10 - Équilibré',
        'n_estimators': 250,
        'max_depth': 25,
        'min_samples_split': 8,
        'min_samples_leaf': 3,
        'max_features': 'sqrt'
    }
]

# Stocker les résultats
results = []

# Tester chaque configuration
for i, config in enumerate(configurations, 1):
    print(f"\n{'='*80}")
    print(f"Configuration {i}/{len(configurations)} : {config['name']}")
    print(f"{'='*80}")

    # Afficher les paramètres
    print(f"\n   Paramètres :")
    for key, value in config.items():
        if key != 'name':
            print(f"      {key:<20} : {value}")

    # Créer le modèle
    model = RandomForestRegressor(
        n_estimators=config['n_estimators'],
        max_depth=config['max_depth'],
        min_samples_split=config['min_samples_split'],
        min_samples_leaf=config['min_samples_leaf'],
        max_features=config['max_features'],
        random_state=42,
        n_jobs=-1,
        verbose=0  # Désactiver verbose pour ne pas polluer
    )

    # Entraîner
    start_time = time.time()
    model.fit(X_train_scaled, y_train)
    training_time = time.time() - start_time

    # Prédictions
    y_train_pred = model.predict(X_train_scaled)
    y_val_pred = model.predict(X_val_scaled)

    # Métriques Train
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)

    # Métriques Validation
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    val_mae = mean_absolute_error(y_val, y_val_pred)
    val_r2 = r2_score(y_val, y_val_pred)

    # Afficher les résultats
    print(f"\n   Résultats :")
    print(f"      Train      : RMSE={train_rmse:.4f} | MAE={train_mae:.4f} | R²={train_r2:.4f}")
    print(f"      Validation : RMSE={val_rmse:.4f} | MAE={val_mae:.4f} | R²={val_r2:.4f}")
    print(f"      Diff R²    : {abs(train_r2 - val_r2):.4f}")
    print(f"      Temps      : {training_time:.2f}s")

    # Stocker
    results.append({
        'Config': config['name'],
        'n_estimators': config['n_estimators'],
        'max_depth': config['max_depth'],
        'min_samples_split': config['min_samples_split'],
        'min_samples_leaf': config['min_samples_leaf'],
        'max_features': config['max_features'],
        'Train_R²': train_r2,
        'Val_R²': val_r2,
        'Val_RMSE': val_rmse,
        'Val_MAE': val_mae,
        'Diff_R²': abs(train_r2 - val_r2),
        'Temps (s)': training_time
    })

# Créer un DataFrame
results_df = pd.DataFrame(results)

# Trier par R² validation (décroissant)
results_df_sorted = results_df.sort_values('Val_R²', ascending=False)

# Top 3
print("\n" + "="*80)
print("TOP 3 DES MEILLEURES CONFIGURATIONS")
print("="*80)

for idx, (i, row) in enumerate(results_df_sorted.head(3).iterrows(), 1):
    print(f"\n{idx}. {row['Config']}")
    print(f"   {'─'*70}")
    print(f"   Paramètres : n_est={row['n_estimators']}, depth={row['max_depth']}, "
          f"min_split={row['min_samples_split']}, min_leaf={row['min_samples_leaf']}, "
          f"max_feat={row['max_features']}")
    print(f"   Val R²     : {row['Val_R²']:.4f}")
    print(f"   Val RMSE   : {row['Val_RMSE']:.4f}")
    print(f"   Val MAE    : {row['Val_MAE']:.4f}")
    print(f"   Overfitting: {row['Diff_R²']:.4f}")
    print(f"   Temps      : {row['Temps (s)']:.2f}s")


TEST DE MULTIPLES CONFIGURATIONS RANDOM FOREST

Configuration 1/10 : Config 1 - Baseline

   Paramètres :
      n_estimators         : 100
      max_depth            : 20
      min_samples_split    : 10
      min_samples_leaf     : 5
      max_features         : sqrt

   Résultats :
      Train      : RMSE=2.0613 | MAE=1.6409 | R²=0.5140
      Validation : RMSE=2.1896 | MAE=1.7489 | R²=0.4532
      Diff R²    : 0.0607
      Temps      : 0.28s

Configuration 2/10 : Config 2 - Plus d'arbres

   Paramètres :
      n_estimators         : 200
      max_depth            : 20
      min_samples_split    : 10
      min_samples_leaf     : 5
      max_features         : sqrt

   Résultats :
      Train      : RMSE=2.0597 | MAE=1.6400 | R²=0.5147
      Validation : RMSE=2.1878 | MAE=1.7483 | R²=0.4541
      Diff R²    : 0.0606
      Temps      : 0.53s

Configuration 3/10 : Config 3 - Arbres profonds

   Paramètres :
      n_estimators         : 200
      max_depth            : 30
      min_samples

Nous allons garder la config numéro 7

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import joblib
import os

print("="*80)
print("ENTRAÎNEMENT DU MODÈLE FINAL")
print("="*80)

# Config 7 - Votre configuration retenue
model_final = RandomForestRegressor(
    n_estimators=200,
    max_depth=30,
    min_samples_split=13,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("\nConfiguration retenue (Config 7) :")
print(f"   n_estimators      : 200")
print(f"   max_depth         : 30")
print(f"   min_samples_split : 13")
print(f"   min_samples_leaf  : 4")
print(f"   max_features      : sqrt")

# Entraîner sur train
print(f"\nEntraînement sur {len(X_train_scaled)} exemples...")
model_final.fit(X_train_scaled, y_train)
print("Entraînement terminé !")

# Évaluation sur train et validation
y_train_pred = model_final.predict(X_train_scaled)
y_val_pred = model_final.predict(X_val_scaled)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
val_mae = mean_absolute_error(y_val, y_val_pred)
val_r2 = r2_score(y_val, y_val_pred)

print("\nRÉSULTATS :")
print(f"\nTrain :")
print(f"   RMSE : {train_rmse:.4f}")
print(f"   MAE  : {train_mae:.4f}")
print(f"   R²   : {train_r2:.4f}")

print(f"\nValidation :")
print(f"   RMSE : {val_rmse:.4f}")
print(f"   MAE  : {val_mae:.4f}")
print(f"   R²   : {val_r2:.4f}")

print(f"\nDifférence R² : {abs(train_r2 - val_r2):.4f}")

# Évaluation FINALE sur TEST
print("\n" + "="*80)
print("ÉVALUATION FINALE SUR L'ENSEMBLE DE TEST")
print("="*80)

y_test_pred = model_final.predict(X_test_scaled)

test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"\nRésultats sur TEST :")
print(f"   RMSE : {test_rmse:.4f}")
print(f"   MAE  : {test_mae:.4f}")
print(f"   R²   : {test_r2:.4f}")

print("\n" + "="*80)
print("RÉSUMÉ FINAL")
print("="*80)
print(f"\n{'Ensemble':<15} {'RMSE':<15} {'MAE':<15} {'R²':<15}")
print("-"*60)
print(f"{'Train':<15} {train_rmse:<15.4f} {train_mae:<15.4f} {train_r2:<15.4f}")
print(f"{'Validation':<15} {val_rmse:<15.4f} {val_mae:<15.4f} {val_r2:<15.4f}")
print(f"{'Test':<15} {test_rmse:<15.4f} {test_mae:<15.4f} {test_r2:<15.4f}")

# Sauvegarder le modèle
os.makedirs('../models', exist_ok=True)

joblib.dump(model_final, '../models/wine_model_final.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(label_encoders, '../models/label_encoders.pkl') if 'label_encoders' in locals() else None

print(f"\nModèle sauvegardé : ../models/wine_model_final.pkl")
print(f"Scaler sauvegardé : ../models/scaler.pkl")

# Sauvegarder les métriques
metrics = {
    'train': {'rmse': train_rmse, 'mae': train_mae, 'r2': train_r2},
    'validation': {'rmse': val_rmse, 'mae': val_mae, 'r2': val_r2},
    'test': {'rmse': test_rmse, 'mae': test_mae, 'r2': test_r2}
}

joblib.dump(metrics, '../models/final_metrics.pkl')
print(f"💾 Métriques sauvegardées : ../models/final_metrics.pkl")